# Fase 3: Integración RAG + Fine-Tuning y Evaluación
### Tutor Analítico Híbrido (RAG + Fine-Tuning)

Este cuaderno integra el recuperador vectorial local **ChromaDB** diseñado en la Fase 1 con el motor lingüístico **Fine-Tuned** mediante LoRA. Utilizaremos el pipeline para someter al Tutor Híbrido al **Banco de Pruebas** (Nivel 1, 2 y 3) para evaluar su efectividad combinada.

In [1]:
# 1. Instalar dependencias mínimas para inferencia
!pip install chromadb sentence-transformers transformers peft torch pymupdf -q

### 2. Cargar Recuperador Vectorial (RAG) y Modelo Lingüístico
Inicializamos la base de datos de ChromaDB y cargamos los adaptadores LoRA sobre el modelo base.

In [2]:
import os
import builtins
import pathlib

# --- MONKEYPATCH PARA EVITAR UNICODEDECODEERROR EN WINDOWS ---
original_open = builtins.open
def patched_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode:
        if 'encoding' in kwargs:
            if kwargs['encoding'] is None:
                kwargs['encoding'] = 'utf-8'
        elif len(args) < 4:
            kwargs['encoding'] = 'utf-8'
        elif len(args) >= 4 and args[3] is None:
            args_list = list(args)
            args_list[3] = 'utf-8'
            args = tuple(args_list)
    return original_open(*args, **kwargs)
builtins.open = patched_open

original_read_text = pathlib.Path.read_text
def patched_read_text(self, encoding=None, errors=None, newline=None):
    return original_read_text(self, encoding=encoding or 'utf-8', errors=errors, newline=newline)
pathlib.Path.read_text = patched_read_text

if hasattr(pathlib, 'PathBase'):
    original_read_text_base = pathlib.PathBase.read_text
    def patched_read_text_base(self, encoding=None, errors=None, newline=None):
        return original_read_text_base(self, encoding=encoding or 'utf-8', errors=errors, newline=newline)
    pathlib.PathBase.read_text = patched_read_text_base
# --------------------------------------------------------------------

import torch
import chromadb
from chromadb.utils import embedding_functions
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Cargar RAG (ChromaDB)
chroma_client = chromadb.PersistentClient(path="data/vectorstore")
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
collection = chroma_client.get_collection(name="tutor_seguridad_corpus", embedding_function=embedding_func)

def retrieve_rag_context(query, top_k=4):
    """Busca fragmentos relevantes y concatena el texto formateado."""
    results = collection.query(query_texts=[query], n_results=top_k)
    context_blocks = []
    if results and results['documents']:
        for i in range(len(results['documents'][0])):
            text = results['documents'][0][i]
            meta = results['metadatas'][0][i]
            context_blocks.append(f"Documento: {meta.get('source')}, Pág/Hoja: {meta.get('page')}\nContenido: {text}")
    return "\n\n---\n\n".join(context_blocks)

# 2. Cargar Modelo Base y Adaptadores LoRA
# Cambiado a 'unsloth/Llama-3.2-1B-Instruct' para coincidir perfectamente con tu entrenamiento de la Fase 2
base_model_name = "unsloth/Llama-3.2-1B-Instruct"
lora_path = "./lora_tutor_seguridad" # Ruta donde se guardaron los adaptadores LoRA

print(f"Cargando tokenizador oficial de: {base_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Configuración de 4 bits para garantizar carga ultra liviana en tu GPU de 4GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

if os.path.exists(lora_path):
    print(" Cargando adaptadores LoRA entrenados sobre modelo base cuantizado en 4-bit...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
    # Acoplar los adaptadores LoRA correspondientes al modelo de 1B
    model = PeftModel.from_pretrained(base_model, lora_path)
    print("¡Tutor Híbrido (RAG + Fine-Tuning LoRA) cargado y acoplado con éxito!")
else:
    print(" ADVERTENCIA: No se encontraron adaptadores LoRA. Cargando modelo base por defecto en 4-bit...")
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )


c:\Users\crist\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3794.71it/s]


Cargando tokenizador oficial de: unsloth/Llama-3.2-1B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


 Cargando adaptadores LoRA entrenados sobre modelo base cuantizado en 4-bit...


Loading weights: 100%|██████████| 146/146 [00:03<00:00, 48.12it/s]


¡Tutor Híbrido (RAG + Fine-Tuning LoRA) cargado y acoplado con éxito!


### 3. Pipeline Híbrido (RAG + Inferencia)
Definimos la función que inyecta la identidad del tutor (System Prompt), recupera la información real del RAG (Contexto) y genera la respuesta final con el comportamiento pedagógico del tutor.

In [3]:
sys_prompt = (
    "Eres un Tutor Analítico especializado en seguridad pública y violencia en México. Tu comportamiento debe ser "
    "estrictamente pedagógico, académico y riguroso. Reglas fundamentales: (1) Cita siempre las fuentes del corpus "
    "al final de cada respuesta usando el formato [Documento X, Pág. Y]. (2) Mantén neutralidad y objetividad ante "
    "temas sensibles. (3) Si el corpus no contiene información suficiente, responde explícitamente que no puedes confirmar "
    "ese dato. (4) Usa el método socrático para guiar al usuario en análisis complejos. (5) Nunca inventes datos, cifras o fuentes."
)

def query_analytical_tutor(user_query, return_diagnostics=False):
    import time
    
    # 1. Recuperar contexto semántico del RAG (ChromaDB) y medir latencia del Vector Store
    start_retrieval = time.time()
    results = collection.query(query_texts=[user_query], n_results=3)
    retrieval_latency = time.time() - start_retrieval
    
    context_blocks = []
    raw_chunks = []
    if results and results['documents']:
        for i in range(len(results['documents'][0])):
            text = results['documents'][0][i]
            meta = results['metadatas'][0][i]
            context_blocks.append(f"Documento: {meta.get('source')}, Pág/Hoja: {meta.get('page')}\nContenido: {text}")
            
            # Guardamos los chunks recuperados con sus metadatos
            raw_chunks.append({
                "source": meta.get('source'),
                "page": meta.get('page'),
                "content": text
            })
    retrieved_context = "\n\n---\n\n".join(context_blocks)
    
    # 2. Configurar el System Prompt con las directrices del Tutor Académico y el Contexto
    sys_prompt_eval = (
        f"{sys_prompt}\n\n"
        "INSTRUCCIÓN DE FUENTES: Basándote estrictamente en el CONTEXTO EXTRAÍDO DEL CORPUS REAL provisto, "
        "responde la duda del usuario y finaliza tu respuesta agregando SIEMPRE una sección llamada "
        "'**Fuentes:** [Nombre_Documento, Pág_Número]' usando los metadatos exactos de los documentos."
    )
    
    messages = [
        {"role": "system", "content": f"{sys_prompt_eval}\n\nCONTEXTO EXTRAÍDO DEL CORPUS:\n{retrieved_context}"},
        {"role": "user", "content": user_query}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Definir los terminadores del chat
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]
    
    # 3. Generar la respuesta del LLM y medir su latencia de generación
    start_generation = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.1,             
            top_p=0.9,
            repetition_penalty=1.15,
            eos_token_id=terminators
        )
    generation_latency = time.time() - start_generation
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    if return_diagnostics:
        return response.strip(), raw_chunks, {
            "retrieval_latency": retrieval_latency,
            "generation_latency": generation_latency,
            "total_latency": retrieval_latency + generation_latency
        }
    return response.strip()


### 4. Banco de Pruebas de Evaluación del Tutor Híbrido
Evaluamos el Tutor con los tres niveles definidos para el entregable.

In [4]:
test_bank = {
    "Nivel 1: Extracción de Datos Directos (Factoid)": [
        ("Q1", "¿Cuáles son las tres entidades federativas con mayor índice de homicidios dolosos según los datos más recientes incluidos en el corpus?"),
        ("Q2", "¿Qué organizaciones, cárteles o grupos delictivos se mencionan con mayor frecuencia operando en la región de Tierra Caliente?"),
        ("Q3", "¿Cuáles son las cifras oficiales reportadas sobre el desplazamiento forzado interno a causa de la violencia durante el último sexenio documentado?")
    ],
    "Nivel 2: Síntesis y Relación de Conceptos": [
        ("Q4", "Según los documentos, ¿cuáles son las principales causas socioeconómicas que los autores asocian directamente al incremento de la violencia urbana?"),
        ("Q5", "Contrasta las estrategias de seguridad pública mencionadas en el corpus. ¿Qué diferencias de enfoque existen entre la militarización y las políticas de prevención social?"),
        ("Q6", "¿Cómo ha evolucionado la tasa de delitos de extorsión (cobro de piso) a nivel nacional y qué sectores económicos se reportan como los más afectados?"),
        ("Q7", "¿Existe alguna diferencia significativa documentada en los tipos de violencia que experimentan las zonas rurales en comparación con las zonas metropolitanas?")
    ],
    "Nivel 3: Razonamiento Analítico y Limitaciones": [
        ("Q8", "Con base en las posturas de las ONGs y las fuentes gubernamentales presentes en los textos, ¿cuáles son las principales contradicciones o discrepancias en el registro de víctimas?"),
        ("Q9", "¿Qué impacto específico tiene la violencia documentada sobre la tasa de deserción escolar en las zonas de alto conflicto? (Nota: Evalúa si el corpus cubre temas educativos o si el RAG alucina una respuesta)."),
        ("Q10", "A partir de las conclusiones de los autores en el corpus, ¿qué vacíos de información, subregistros o falta de datos fiables se identifican como el principal obstáculo para medir la violencia real en el país?")
    ]
}

# Bucle evaluativo que imprime los resultados estructurados
for level, questions in test_bank.items():
    print(f"\n🔹 **{level}**")
    for q_id, question in questions:
        print(f"\n🔸 **[{q_id}] Pregunta:** {question}")
        
        # Inferencia con diagnósticos (latencias y chunks)
        respuesta, chunks, latencies = query_analytical_tutor(question, return_diagnostics=True)
        
        # 1. Mostrar Tiempos de Latencia
        print(f"\n⏱️ **Tiempos de Latencia:**")
        print(f"  • Búsqueda y Recuperación (RAG): {latencies['retrieval_latency']:.4f} seg")
        print(f"  • Generación de Respuesta (LLM): {latencies['generation_latency']:.4f} seg")
        print(f"  • Latencia Total del Sistema:   {latencies['total_latency']:.4f} seg")
        
        # 2. Mostrar Chunks Recuperados (Justificación de respuesta)
        print(f"\n📚 **Chunks de Texto Recuperados (Contexto Justificativo):**")
        for idx, chunk in enumerate(chunks, 1):
            print(f"  [{idx}] Fuente: {chunk['source']} | Página/Hoja: {chunk['page']}")
            indented_content = "\n".join([f"      {line}" for line in chunk['content'].strip().split('\n')])
            print(f"    • Contenido:\n{indented_content}")
            print(f"    {'~'*60}")
        
        # 3. Mostrar respuesta final generada por el Tutor Híbrido
        print(f"\n🤖 **Respuesta del Tutor Híbrido:**\n{respuesta}\n")



🔹 **Nivel 1: Extracción de Datos Directos (Factoid)**

🔸 **[Q1] Pregunta:** ¿Cuáles son las tres entidades federativas con mayor índice de homicidios dolosos según los datos más recientes incluidos en el corpus?


[transformers] Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏱️ **Tiempos de Latencia:**
  • Búsqueda y Recuperación (RAG): 0.7474 seg
  • Generación de Respuesta (LLM): 28.9734 seg
  • Latencia Total del Sistema:   29.7208 seg

📚 **Chunks de Texto Recuperados (Contexto Justificativo):**
  [1] Fuente: Informe_IncidenciaDelictiva_Fuero_Comun_Abril_2026.pdf | Página/Hoja: 6
    • Contenido:
      HOMICIDIO DOLOSO POR ENTIDAD FEDERATIVA
      Abril 2026
      8 entidad(es) federativa(s) concentran el
      53.0%  (834)
      del total de casos de Homicidio doloso
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  [2] Fuente: vap-anual-dic-2025.pdf | Página/Hoja: 16
    • Contenido:
      el comportamiento es 
      igualmente preocupante: en un estado 
      que concentra una parte sustantiva del 
      homicidio doloso nacional, la expansión 
      de esta categoría apunta a que parte de 
      la violencia letal o cuasi letal podría estar 
      desplazándose hacia tipificaciones resi-
      duales, ya sea por debilidades institu

[transformers] Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏱️ **Tiempos de Latencia:**
  • Búsqueda y Recuperación (RAG): 0.0513 seg
  • Generación de Respuesta (LLM): 30.2476 seg
  • Latencia Total del Sistema:   30.2989 seg

📚 **Chunks de Texto Recuperados (Contexto Justificativo):**
  [1] Fuente: ESTUDIO-RECLUTADOS-POR-LA-DELINCUENCIA-ORGANIZADA.pdf | Página/Hoja: 99
    • Contenido:
      elincuencia organizada son los policías y los agentes de ministerios públicos estatales; 
      igualmente, se señala a los militares y la guardia nacional como autoridades que hacen 
      acuerdos con grupos criminales, pero en menor proporción, y esta también es una 
      tendencia que se repite en las tres zonas, norte, sur y centro.
      La delincuencia organizada opera en casas de seguridad, que son inmuebles 
      estratégicos donde se resguarda la droga, las armas y los productos o mercancías propias 
      de la activi
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  [2] Fuente: ESTUDIO-RECLUTADOS-POR-LA-DELINCUENCIA-ORGANIZ

...


⏱️ **Tiempos de Latencia:**
  • Búsqueda y Recuperación (RAG): 0.0542 seg
  • Generación de Respuesta (LLM): 39.6257 seg
  • Latencia Total del Sistema:   39.6800 seg

📚 **Chunks de Texto Recuperados (Contexto Justificativo):**
  [1] Fuente: violencia_estudiantes_informe.pdf | Página/Hoja: 56
    • Contenido:
      teles privados reportan los mayores porcentajes 
      de violencia. El estudio no proporciona información para determinar si las dife-
      rencias son estadísticamente significativas. Tampoco se proporcionan elemen-
      tos para valorar la relación entre subsistema y grado de violencia, la información 
      sugiere más bien una diferencia entre los subsistemas rurales y los urbanos.
      El estudio incluyó preguntas sobre convivencia con las que se podrían haber realiza-
      do análisis correlacionales, pero no se reporta
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  [2] Fuente: violencia_estudiantes_informe.pdf | Página/Hoja: 86
    • Contenid

[transformers] Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏱️ **Tiempos de Latencia:**
  • Búsqueda y Recuperación (RAG): 0.0586 seg
  • Generación de Respuesta (LLM): 52.0402 seg
  • Latencia Total del Sistema:   52.0988 seg

📚 **Chunks de Texto Recuperados (Contexto Justificativo):**
  [1] Fuente: ESTUDIO-RECLUTADOS-POR-LA-DELINCUENCIA-ORGANIZADA.pdf | Página/Hoja: 7
    • Contenido:
      uentes que coincidan en cifras, es necesario 
      apelar por el reconocimiento de la problemática desde otras aristas. El trabajo de 
      organizaciones no gubernamentales es fundamental, pues su labor debe reconocer y 
      visibilizar las historias de quienes viven esta realidad para detectar y entender los factores 
      que vulneran a esta población y así dar pie a soluciones específicas.
      	
      Dentro de este contexto, Reinserta decide dar voz a un grupo de adolescentes 
      cuyas historias han conducido a que estén
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  [2] Fuente: Manual_metodol_gico_RNID_V1.0_VF.pdf | Pág

[transformers] Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏱️ **Tiempos de Latencia:**
  • Búsqueda y Recuperación (RAG): 0.0533 seg
  • Generación de Respuesta (LLM): 78.9288 seg
  • Latencia Total del Sistema:   78.9821 seg

📚 **Chunks de Texto Recuperados (Contexto Justificativo):**
  [1] Fuente: violencia_estudiantes_informe.pdf | Página/Hoja: 61
    • Contenido:
      61
      3
      Frecuencia de la violencia escolar 
      entre estudiantes en estudios del 
      acervo de Mejoredu
      Después de la presentación del concepto de violencia y de los resultados de estudios 
      con muestras nacionales que han abordado los factores de la violencia, en este capí-
      tulo se aborda la primera parte del propósito central del estudio, a saber: el análisis 
      de la frecuencia de la violencia entre estudiantes en las escuelas.
      Para ello, se recurrió a los reportes de estudios19 disponibles en el acervo
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  [2] Fuente: violencia_estudiantes_informe.pdf | Página/Hoja: 

### 5. Chat Interactivo CLI (Modo Tutor Activo)
Ejecuta esta celda para iniciar una conversación interactiva libre con tu Tutor Analítico.

In [6]:
print("=== Tutor Analítico Especializado en Seguridad (RAG + LoRA) ===")
print("Escribe tu consulta académica o 'salir' para terminar.\n")

while True:
    try:
        query = input("Tu consulta: ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    
    if not query or query.lower() in ('salir', 'exit', 'quit'):
        print("Sesión académica finalizada. Hasta pronto.")
        break
        
    print("\n--- ⏳ PROCESANDO CONSULTA... ---")
    # Realizamos la inferencia con diagnósticos para obtener los chunks y latencias en tiempo real
    respuesta, chunks, latencies = query_analytical_tutor(query, return_diagnostics=True)
    
    # 1. Mostrar tiempos de latencia detallados
    print(f"\n⏱️ **Tiempos de Latencia:**")
    print(f"  • Búsqueda y Recuperación (RAG): {latencies['retrieval_latency']:.4f} seg")
    print(f"  • Generación de Respuesta (LLM): {latencies['generation_latency']:.4f} seg")
    print(f"  • Latencia Total del Sistema:   {latencies['total_latency']:.4f} seg")
    
    # 2. Mostrar los chunks de texto recuperados del Vector Store
    print(f"\n📚 **Chunks de Texto Recuperados (Contexto Justificativo):**")
    for idx, chunk in enumerate(chunks, 1):
        print(f"  [{idx}] Fuente: {chunk['source']} | Página/Hoja: {chunk['page']}")
        indented_content = "\n".join([f"      {line}" for line in chunk['content'].strip().split('\n')])
        print(f"    • Contenido:\n{indented_content}")
        print(f"    {'~'*60}")
        
    # 3. Mostrar la respuesta final pedagógica del Tutor
    print(f"\n🤖 **Respuesta del Tutor Híbrido:**\n{respuesta}")
    print(f"\n{'='*80}\n")


=== Tutor Analítico Especializado en Seguridad (RAG + LoRA) ===
Escribe tu consulta académica o 'salir' para terminar.


--- ⏳ PROCESANDO CONSULTA... ---


[transformers] Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏱️ **Tiempos de Latencia:**
  • Búsqueda y Recuperación (RAG): 0.0663 seg
  • Generación de Respuesta (LLM): 16.5819 seg
  • Latencia Total del Sistema:   16.6482 seg

📚 **Chunks de Texto Recuperados (Contexto Justificativo):**
  [1] Fuente: page-one-noviembre.pdf | Página/Hoja: 2
    • Contenido:
      .7%
      358.0%
      268.6%
      75.5%
      Números absolutos de delitos asociados a la violencia letal 
      a nivel nacional, 2015-2025 (enero-octubre)
      Fuente: México Evalúa datos del SESNSP (víctimas en carpetas) y RNPDNO para el periodo de enero a octubre de cada año. 
      Datos de personas desaparecidas y no localizadas consultados el 13 de noviembre de 2025. 
      Ver apunte metodológico al ﬁnal de este documento.
      Violencia letal 
      (considerando todos 
      los proxys anteriores)
      Homicidio doloso
      Homicidio culposo
      Feminicidio
      Pe
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  [2] Fuente: page-one-noviembre.pdf |